# Optuna — Búsqueda de hiperparámetros LDA (Cercanías)

**Objetivo:** maximizar una métrica compuesta:

```
score = per_station_loglik  +  0.5 × active_ratio
```

- `per_station_loglik = lda_.score(X) / n_estaciones` — log-likelihood variacional normalizado por estación. Mide cuánto explica el modelo los datos sin favorecer mecánicamente K grandes.
- `active_ratio = n_tópicos_activos / n_topics` — fracción de tópicos en los que al menos una estación tiene peso > 10 %. Penaliza los tópicos muertos (capacidad desperdiciada para el pipeline MDLF).

**Estrategia de pruning:** `MedianPruner` con evaluación en **2 pasos** por trial:
- Paso 0 (rápido): fit con `max_iter=15`, `n_init=1` (~1 s) → el pruner descarta configs claramente malas.
- Paso 1 (completo): fit con el `max_iter` del trial, `n_init=1` → score final.

```
Hiperparámetros buscados
──────────────────────────────────────────────────────
 n_topics          int        [4, 32]    step=2
 alpha             float      [0.01, 1.0]  log-scale   ← límite sklearn
 beta              float      [0.001, 0.5] log-scale
 use_periods       categórico [True, 4, False]
 token_granularity categórico ['temporal', 'station']
 log_counts        categórico [False, True]
 max_iter          int        [50, 300]   step=50
```

In [ ]:
import subprocess, sys
subprocess.run(
    [sys.executable, "-m", "pip", "install", "optuna", "--quiet"],
    check=True
)
import optuna
print(f'Optuna {optuna.__version__} listo')

In [ ]:
import sys, time, logging, warnings
from itertools import product as iproduct
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA
import plotly.graph_objects as go
from IPython.display import HTML

import optuna
from optuna.pruners import MedianPruner
from optuna.samplers import TPESampler

optuna.logging.set_verbosity(optuna.logging.WARNING)
logging.getLogger().setLevel(logging.WARNING)
warnings.filterwarnings('ignore')

sys.path.insert(0, '.')
from region_function_lda import RegionFunctionLDA

print('Imports OK')

In [ ]:
MATRICES_DIR = Path('../../cubo_izquierdo/matrices')
OD_FOLDERS = [
    'validaciones_3_7_febrero'
]

frames = []
for folder in OD_FOLDERS:
    df = pd.read_csv(MATRICES_DIR / folder / 'matriz_od.csv')
    frames.append(df)
    print(f'  {folder}: {len(df):,} filas')

od = pd.concat(frames, ignore_index=True)
print(f'\nTotal: {len(od):,} filas')

# Mapa punto_acceso → nombre para etiquetas
nombres = dict(zip(od['origen_punto_acceso'], od['origen_nombre']))
nombres.update(dict(zip(od['destino_punto_acceso'], od['destino_nombre'])))

def label(sid, max_len=30):
    n = nombres.get(sid, '')
    return f'{n[:max_len]}  ({sid})' if n else sid

In [ ]:
# Pre-computar las 12 matrices doc-term ANTES de Optuna.
#
# El preprocessing de 58M filas (copy, to_datetime, groupby, vectorize)
# tarda ~10-20s por combinación. Sin esto, se repetiría 2 veces por trial
# × 50 trials = 100 veces. Así se hace solo 12 veces en total.
#
# La clave de caché es (use_periods, token_granularity, log_counts):
# los únicos parámetros que cambian el vocabulario. Alpha, beta, n_topics
# y max_iter solo afectan al entrenamiento LDA, no al preprocesado.

X_CACHE: dict = {}

combos = list(iproduct([True, 4, False], ['temporal', 'station'], [False, True]))
print(f'Pre-computando {len(combos)} matrices doc-term...\n')

for use_periods, token_granularity, log_counts in combos:
    key = (use_periods, token_granularity, log_counts)
    t0  = time.time()

    # Usar RegionFunctionLDA solo para construir documentos y vectorizar
    m_tmp = RegionFunctionLDA(
        n_topics=4, use_periods=use_periods, token_granularity=token_granularity,
        log_counts=log_counts, filter_prefixes=['04'], slot_minutes=15,
        max_iter=1, n_init=1, min_df=1, random_state=42,
    )
    od_f  = m_tmp._filter(od)
    docs  = m_tmp.build_documents(od_f)
    docs  = m_tmp._filter_vocab(docs)          # min_df=1, df_smoothing=False → sin cambios
    stns  = sorted(docs.keys())
    vec   = DictVectorizer(sparse=True, dtype=np.float64)
    X     = vec.fit_transform([docs.get(s, {}) for s in stns])

    X_CACHE[key] = {'stations': stns, 'vectorizer': vec, 'X': X}

    print(
        f'  use_periods={str(use_periods):<5}  '
        f'gran={token_granularity:<9}  '
        f'log={str(log_counts):<5}  →  '
        f'{len(stns)} estaciones  {X.shape[1]:>5} tokens  '
        f'({time.time()-t0:.1f}s)'
    )

print(f'\nCache listo: {len(X_CACHE)} combinaciones')

---
## Función objetivo con pruning en 2 pasos

In [ ]:
ACTIVE_TOPIC_THRESHOLD = 0.10
ACTIVE_RATIO_WEIGHT    = 0.5


def fit_lda(X, n_topics, alpha, beta, max_iter, random_state=42):
    """Entrena LDA sobre X pre-computada y devuelve (lda, theta_arr)."""
    lda = LatentDirichletAllocation(
        n_components=n_topics,
        doc_topic_prior=alpha,
        topic_word_prior=beta,
        learning_method='batch',
        max_iter=max_iter,
        evaluate_every=-1,
        random_state=random_state,
    )
    lda.fit(X)
    theta_arr = lda.transform(X)
    return lda, theta_arr


def compute_score(lda, X, theta_arr, n_topics):
    """Devuelve (score_total, per_station_loglik, active_ratio)."""
    n_stations         = theta_arr.shape[0]
    per_station_loglik = lda.score(X) / n_stations
    n_active           = int((theta_arr.max(axis=0) > ACTIVE_TOPIC_THRESHOLD).sum())
    active_ratio       = n_active / n_topics
    score              = per_station_loglik + ACTIVE_RATIO_WEIGHT * active_ratio
    return score, per_station_loglik, active_ratio


def make_theta_df(theta_arr, stations, n_topics):
    cols = [f'topic_{i}' for i in range(n_topics)]
    return pd.DataFrame(theta_arr, index=stations, columns=cols)


def make_model(n_topics, alpha, beta, use_periods, token_granularity,
               log_counts, max_iter, n_init, random_state=42):
    """Crea un RegionFunctionLDA completo (solo para el modelo final)."""
    return RegionFunctionLDA(
        n_topics=n_topics, alpha=alpha, beta=beta,
        use_periods=use_periods, token_granularity=token_granularity,
        log_counts=log_counts, max_iter=max_iter, n_init=n_init,
        filter_prefixes=['04'], slot_minutes=15, random_state=random_state,
    )


def objective(trial: optuna.Trial) -> float:
    n_topics          = trial.suggest_int("n_topics",          4,    32,   step=2)
    alpha             = trial.suggest_float("alpha",           0.01, 1.0,  log=True)
    beta              = trial.suggest_float("beta",            0.001, 0.5, log=True)
    use_periods       = trial.suggest_categorical("use_periods",       [True, 4, False])
    token_granularity = trial.suggest_categorical("token_granularity", ['temporal', 'station'])
    log_counts        = trial.suggest_categorical("log_counts",        [False, True])
    max_iter          = trial.suggest_int("max_iter",          50,   300,  step=50)

    # Recuperar X pre-computada — sin tocar los 58M filas
    cache = X_CACHE[(use_periods, token_granularity, log_counts)]
    X, stations = cache['X'], cache['stations']

    # Paso 0: fit rápido para pruning (~1 s con max_iter=15)
    lda_q, theta_q = fit_lda(X, n_topics, alpha, beta, max_iter=15)
    score_q, _, _  = compute_score(lda_q, X, theta_q, n_topics)

    trial.report(score_q, step=0)
    if trial.should_prune():
        raise optuna.TrialPruned()

    # Paso 1: fit completo
    lda_f, theta_f             = fit_lda(X, n_topics, alpha, beta, max_iter=max_iter)
    score_f, loglik_f, active_f = compute_score(lda_f, X, theta_f, n_topics)

    trial.report(score_f, step=1)
    trial.set_user_attr('per_station_loglik', round(loglik_f, 4))
    trial.set_user_attr('active_ratio',       round(active_f, 4))
    trial.set_user_attr('n_active_topics',    int(round(active_f * n_topics)))
    trial.set_user_attr('vocab_size',         int(X.shape[1]))

    return score_f

---
## Ejecutar la búsqueda

- **50 trials** — cada trial tarda ~10-20 s; con pruning en paso 0 se recorta bastante.
- `n_startup_trials=10`: los primeros 10 trials completos dan referencia al pruner.
- `n_warmup_steps=0`: el pruner actúa desde el paso 0 (fit rápido).

In [ ]:
def trial_callback(study: optuna.Study, trial: optuna.trial.FrozenTrial) -> None:
    if trial.state == optuna.trial.TrialState.PRUNED:
        print(f"  Trial {trial.number:>3} | PODADO en paso {trial.last_step}")
    elif trial.state == optuna.trial.TrialState.COMPLETE:
        attrs = trial.user_attrs
        print(
            f"  Trial {trial.number:>3} | score={trial.value:>8.4f} | "
            f"loglik={attrs.get('per_station_loglik', float('nan')):>9.4f} | "
            f"active={attrs.get('n_active_topics', '?'):>2}/{trial.params['n_topics']:>2} | "
            f"K={trial.params['n_topics']:>2}  "
            f"α={trial.params['alpha']:.4f}  β={trial.params['beta']:.4f}  "
            f"gran={trial.params['token_granularity']:<9}  "
            f"periods={str(trial.params['use_periods']):<5}  "
            f"log={trial.params['log_counts']}"
        )


pruner  = MedianPruner(n_startup_trials=10, n_warmup_steps=0, interval_steps=1)
sampler = TPESampler(seed=42)

study = optuna.create_study(
    direction="maximize",
    sampler=sampler,
    pruner=pruner,
    study_name="LDA_cercanias",
)

print(f"{'Trial':>7}  {'score':>8}  {'loglik':>9}  {'active':>6}  params")
print("-" * 90)
study.optimize(objective, n_trials=50, callbacks=[trial_callback], show_progress_bar=True)

complete = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
pruned   = [t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED]

print(f'\nTrials completados : {len(complete)}')
print(f'Trials podados     : {len(pruned)}')
print(f'\nMejor score        : {study.best_value:.4f}')
print(f'Mejores params:')
for k, v in study.best_params.items():
    print(f'  {k:<22} = {v}')
print(f'  {"n_active_topics":<22} = {study.best_trial.user_attrs.get("n_active_topics", "?")} / {study.best_params["n_topics"]}')

---
## Análisis de resultados

In [ ]:
top10 = sorted(complete, key=lambda t: t.value, reverse=True)[:10]

rows = [{
    'rank':             i + 1,
    'score':            round(t.value, 4),
    'per_sta_loglik':   t.user_attrs.get('per_station_loglik', float('nan')),
    'active_ratio':     t.user_attrs.get('active_ratio',       float('nan')),
    'n_active':         t.user_attrs.get('n_active_topics',    '?'),
    'vocab_size':       t.user_attrs.get('vocab_size',         '?'),
    **t.params,
} for i, t in enumerate(top10)]

df_top = pd.DataFrame(rows)
print('Top-10 configuraciones:')
display(df_top)

In [ ]:
vals        = [t.value for t in complete]
best_so_far = np.maximum.accumulate(vals)

fig, ax = plt.subplots(figsize=(10, 4))
ax.scatter(range(len(vals)), vals, s=18, alpha=0.6, label='Score trial')
ax.plot(range(len(vals)), best_so_far, color='red', linewidth=1.5, label='Mejor hasta ahora')
ax.set_xlabel('Trial completado')
ax.set_ylabel('Score')
ax.set_title('Optuna LDA Cercanías — historia de optimización')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
try:
    importance  = optuna.importance.get_param_importances(study)
    params_imp  = list(importance.keys())
    values_imp  = list(importance.values())

    fig, ax = plt.subplots(figsize=(7, 4))
    bars = ax.barh(params_imp[::-1], values_imp[::-1], color='steelblue', alpha=0.85)
    ax.set_xlabel('Importancia relativa (fANOVA)')
    ax.set_title('Importancia de hiperparámetros — LDA Cercanías')
    for bar, v in zip(bars, values_imp[::-1]):
        ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height() / 2,
                f'{v:.3f}', va='center', fontsize=8)
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f'Importancia no disponible: {e}')

In [ ]:
# Score vs cada hiperparámetro continuo
cont_params = ['n_topics', 'alpha', 'beta', 'max_iter']
fig, axes   = plt.subplots(1, len(cont_params), figsize=(14, 4))

for ax, pname in zip(axes, cont_params):
    xs = [t.params[pname] for t in complete]
    ys = [t.value         for t in complete]
    ax.scatter(xs, ys, s=20, alpha=0.6, color='steelblue')
    ax.axhline(study.best_value, color='red', linewidth=0.8, linestyle='--', alpha=0.7)
    ax.scatter([study.best_params[pname]], [study.best_value],
               color='red', s=70, zorder=5)
    ax.set_xlabel(pname, fontsize=9)
    ax.set_ylabel('Score', fontsize=8)
    if pname in ('alpha', 'beta'):
        ax.set_xscale('log')

fig.suptitle('Hiperparámetros continuos vs Score — LDA Cercanías', fontsize=10)
plt.tight_layout()
plt.show()

# Score por valor de hiperparámetros categóricos
cat_params = ['use_periods', 'token_granularity', 'log_counts']
fig, axes  = plt.subplots(1, len(cat_params), figsize=(12, 4))

for ax, pname in zip(axes, cat_params):
    groups = {}
    for t in complete:
        k = str(t.params[pname])
        groups.setdefault(k, []).append(t.value)
    keys = sorted(groups.keys())
    ax.boxplot([groups[k] for k in keys], labels=keys)
    ax.set_title(pname, fontsize=9)
    ax.set_ylabel('Score', fontsize=8)

fig.suptitle('Hiperparámetros categóricos vs Score — LDA Cercanías', fontsize=10)
plt.tight_layout()
plt.show()

---
## Entrenamiento final con la mejor configuración

Se usa `n_init=3` para reducir el riesgo de mala inicialización.

In [ ]:
best = study.best_params
print('Entrenando modelo final (n_init=3)...')
print(best)

t0 = time.time()
model = make_model(
    n_topics=best['n_topics'],
    alpha=best['alpha'],
    beta=best['beta'],
    use_periods=best['use_periods'],
    token_granularity=best['token_granularity'],
    log_counts=best['log_counts'],
    max_iter=best['max_iter'],
    n_init=3,
)
theta = model.fit_transform(od)
score_final, loglik_final, active_final = compute_score(model.lda_, model.X_, theta.values, best['n_topics'])

print(f'\nCompletado en {time.time()-t0:.1f}s')
print(f'Score final:         {score_final:.4f}')
print(f'per_station_loglik:  {loglik_final:.4f}')
print(f'active_ratio:        {active_final:.4f}  ({int(active_final * best["n_topics"])} / {best["n_topics"]} tópicos activos)')
print(f'Vocab size:          {model.X_.shape[1]} tokens')
print(f'Theta shape:         {theta.shape}')

---
## Validación del modelo final

In [ ]:
# Integridad de theta
sumas    = theta.sum(axis=1)
ok_suma  = ((sumas - 1.0).abs() < 1e-4).all()
ok_rango = (theta.values.min() >= 0) and (theta.values.max() <= 1.0)
ok_nan   = not theta.isnull().any().any()

print(f'Filas suman 1:     {"✓" if ok_suma  else "✗"}  (min={sumas.min():.6f}, max={sumas.max():.6f})')
print(f'Valores en [0,1]:  {"✓" if ok_rango else "✗"}  (min={theta.values.min():.6f}, max={theta.values.max():.6f})')
print(f'Sin NaN:           {"✓" if ok_nan   else "✗"}')

# Distribución de tópico dominante
topic_cols = [f'topic_{i}' for i in range(model.n_topics)]
dom        = theta.idxmax(axis=1).value_counts().reindex(topic_cols, fill_value=0)
pct_max    = dom.max() / len(theta) * 100
n_dead     = (dom == 0).sum()

print(f'\nEstaciones por tópico dominante  (max concentración: {pct_max:.0f}%)')
print(f'Tópicos sin estación dominante (muertos): {n_dead} / {model.n_topics}')
print(dom.to_string())

In [ ]:
# Heatmap theta (estaciones × tópicos), ordenado por tópico dominante
th       = theta[topic_cols].copy()
dom_num  = th.idxmax(axis=1).str.extract(r'(\d+)').astype(int)
dom_peso = th.max(axis=1)

order_idx = (
    pd.DataFrame({'_num': dom_num.values.ravel(), '_peso': dom_peso.values}, index=th.index)
    .sort_values(['_num', '_peso'], ascending=[True, False])
    .index
)
th_sorted = th.loc[order_idx]
n         = len(th_sorted)

fig, ax = plt.subplots(figsize=(11, max(6, n * 0.25)))
im = ax.imshow(th_sorted.values, aspect='auto', cmap='YlOrRd', vmin=0, vmax=th_sorted.values.max())

ax.set_title(
    f'Theta — Cercanías  |  {model.n_topics} tópicos  |  '
    f'α={best["alpha"]:.3f}  β={best["beta"]:.4f}\n'
    f'granularidad={best["token_granularity"]}  use_periods={best["use_periods"]}  '
    f'log_counts={best["log_counts"]}  ({n} estaciones)',
    fontsize=9
)
ax.set_xlabel('Tópico')
ax.set_xticks(range(model.n_topics))
ax.set_xticklabels([f'T{i}' for i in range(model.n_topics)], fontsize=8)
ax.set_yticks(range(n))
ax.set_yticklabels(
    [f'{nombres.get(s, s)[:28]}  ({s})' for s in th_sorted.index],
    fontsize=6
)
plt.colorbar(im, ax=ax, label='Peso del tópico', shrink=0.5)

prev = dom_num.loc[order_idx].values.ravel()[0]
for k, v in enumerate(dom_num.loc[order_idx].values.ravel()):
    if v != prev:
        ax.axhline(k - 0.5, color='white', linewidth=1.2, alpha=0.8)
        prev = v

plt.tight_layout()
plt.show()

In [ ]:
# Perfil IN/OUT por franja temporal de cada tópico
BIN_ORDER_PERIOD  = ['noche', 'madrug', 'punta_man', 'mediodia', 'tarde', 'punta_tar', 'noche_ini']
BIN_ORDER_4       = ['night', 'morning', 'afternoon', 'evening']
BIN_ORDER_NUMERIC = [f'k{str(i).zfill(2)}' for i in range(96 // (model.slot_minutes // 15))]

if best['use_periods'] is True:
    BIN_ORDER = BIN_ORDER_PERIOD
elif best['use_periods'] == 4:
    BIN_ORDER = BIN_ORDER_4
else:
    BIN_ORDER = BIN_ORDER_NUMERIC


def parse_token(token):
    """Extrae (dirección, franja) de un token, sea cual sea la granularidad."""
    direction = 'OUT' if token.startswith('OUT') else 'IN'
    inner     = token[len(direction) + 1:]   # todo lo que sigue al prefijo OUT_ / IN_
    # El daytype siempre es el último segmento (_WD o _WE)
    # La franja puede ser 'punta_man', 'k08', 'morning', etc.
    # Para granularidad station/line hay ID de estación antes de la franja
    if best['token_granularity'] == 'temporal':
        franja = inner.rsplit('_', 1)[0]
    elif best['token_granularity'] == 'line':
        # OUT_L1_morning_WD → inner='L1_morning_WD' → franja='morning'
        parts  = inner.split('_')
        franja = '_'.join(parts[1:-1])
    else:  # station
        # OUT_02_L1_P5_morning_WD → inner='02_L1_P5_morning_WD'
        # el ID tiene 3 partes (02, L1, P5) luego franja y daytype
        parts  = inner.split('_')
        franja = '_'.join(parts[3:-1])
    return direction, franja


topics_df = model.inspect_topics(n_top_tokens=50)
n_topics  = model.n_topics
cols_grid = min(4, n_topics)
rows_grid = (n_topics + cols_grid - 1) // cols_grid

fig, axes = plt.subplots(rows_grid, cols_grid, figsize=(cols_grid * 4, rows_grid * 3))
axes_flat = axes.flatten() if n_topics > 1 else [axes]

for i, (topic, row) in enumerate(topics_df.iterrows()):
    bins_out, bins_in = {}, {}
    for token, peso in zip(row['top_tokens'], row['top_weights']):
        try:
            dir_, franja = parse_token(token)
        except Exception:
            continue
        if dir_ == 'OUT':
            bins_out[franja] = bins_out.get(franja, 0) + peso
        else:
            bins_in[franja]  = bins_in.get(franja,  0) + peso

    ax       = axes_flat[i]
    all_keys = [b for b in BIN_ORDER if b in bins_out or b in bins_in]
    if not all_keys:
        all_keys = sorted(set(bins_out) | set(bins_in))
    x_pos = range(len(all_keys))
    out_v = [bins_out.get(b, 0) for b in all_keys]
    in_v  = [bins_in.get(b,  0) for b in all_keys]
    w = 0.38
    ax.bar([x - w/2 for x in x_pos], out_v, width=w, color='steelblue', alpha=0.85, label='OUT')
    ax.bar([x + w/2 for x in x_pos], in_v,  width=w, color='darkorange', alpha=0.85, label='IN')
    ax.set_xticks(list(x_pos))
    ax.set_xticklabels(all_keys, rotation=45, ha='right', fontsize=6)
    ax.set_title(topic, fontsize=9)
    if i == 0:
        ax.legend(fontsize=6)

for j in range(i + 1, len(axes_flat)):
    axes_flat[j].set_visible(False)

fig.suptitle(f'Distribución horaria por tópico — Cercanías (azul=OUT, naranja=IN)', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# Estación más representativa por tópico
print(f'Estación más representativa de cada tópico (config óptima):\n')
for col in theta.columns:
    best_sta = theta[col].idxmax()
    val      = theta[col].max()
    nom      = nombres.get(best_sta, '')
    print(f'  {col:>10}: {best_sta}  {nom:<40}  ({val:.3f})')

In [ ]:
# Vecinos más cercanos en espacio LDA
KEY_STATIONS = {
    '04_L0_P108': 'SOL (hub central)',
    '04_L0_P7':   'ATOCHA (gran hub)',
    '04_L0_P13':  'CHAMARTÍN (hub norte)',
    '04_L0_P62':  'PRÍNCIPE PÍO',
    '04_L0_P10':  'CANTOBLANCO UNIV.',
    '04_L0_P11':  'CERCEDILLA (sierra)',
    '04_L0_P57':  'PARLA (sur)',
    '04_L0_P1':   'ALCALÁ DE HENARES',
    '04_L0_P116': 'ROBLEDO DE CHAVELA',
    '04_L0_P77':  'TRES CANTOS',
}
K = 7

X_theta      = theta.values.astype(float)
sim_full     = cosine_similarity(X_theta)
station_list = list(theta.index)

print(f'Top-{K} vecinos en espacio LDA (similitud coseno sobre theta)\n')
for sid, desc in KEY_STATIONS.items():
    if sid not in station_list:
        print(f'  {desc} ({sid}): no encontrada\n'); continue
    idx  = station_list.index(sid)
    sims = sim_full[idx]
    top  = np.argsort(sims)[::-1][1:K+1]
    print(f'{desc}  [{sid}]')
    for rank, j in enumerate(top, 1):
        print(f'  {rank}. {nombres.get(station_list[j], station_list[j]):<38} ({station_list[j]})  sim={sims[j]:.4f}')
    print()

In [ ]:
# Perfiles theta de cada estación clave + sus 3 vecinos más cercanos
K_VIS = 3
x_pos = range(model.n_topics)

for sid, desc in KEY_STATIONS.items():
    if sid not in station_list:
        continue
    idx   = station_list.index(sid)
    sims  = sim_full[idx]
    top   = np.argsort(sims)[::-1][1:K_VIS+1]
    group = [sid] + [station_list[j] for j in top]

    fig, axes = plt.subplots(1, len(group), figsize=(len(group) * 3.5, 3.2), sharey=True)

    for k, (ax, s) in enumerate(zip(axes, group)):
        perfil = theta.loc[s].values
        color  = 'steelblue' if k == 0 else 'tomato'
        ax.bar(x_pos, perfil, color=color, alpha=0.85 if k == 0 else 0.70)
        ax.set_ylim(0, max(theta.values.max() * 1.05, 0.05))
        ax.set_xticks(list(x_pos))
        ax.set_xticklabels([f'T{i}' for i in x_pos], rotation=90, fontsize=6)
        nom   = nombres.get(s, s)
        sim_v = sims[station_list.index(s)]
        ttl   = f'{nom[:22]}\n({s})'
        if k > 0:
            ttl += f'\nsim={sim_v:.3f}'
        ax.set_title(ttl, fontsize=7)

    fig.suptitle(f'{desc} — perfil theta + vecinos más cercanos', fontsize=9)
    plt.tight_layout()
    plt.show()

In [ ]:
# Heatmap similitud coseno entre estaciones (mismo orden que heatmap theta)
X_sorted  = th_sorted.values.astype(float)
sim_sorted = cosine_similarity(X_sorted)

fig, ax = plt.subplots(figsize=(10, 9))
im = ax.imshow(sim_sorted, cmap='RdYlGn', vmin=0, vmax=1, aspect='auto')
ax.set_title('Similitud coseno entre estaciones en espacio LDA\n(ordenadas por tópico dominante)', fontsize=10)
ax.set_xticks(range(n))
ax.set_xticklabels([s for s in th_sorted.index], rotation=90, fontsize=4)
ax.set_yticks(range(n))
ax.set_yticklabels([f'{nombres.get(s, s)[:22]}' for s in th_sorted.index], fontsize=4)
plt.colorbar(im, ax=ax, label='Similitud coseno', shrink=0.5)

prev = dom_num.loc[order_idx].values.ravel()[0]
for k, v in enumerate(dom_num.loc[order_idx].values.ravel()):
    if v != prev:
        ax.axhline(k - 0.5, color='white', linewidth=0.8)
        ax.axvline(k - 0.5, color='white', linewidth=0.8)
        prev = v

plt.tight_layout()
plt.show()

In [ ]:
# PCA interactivo coloreado por tópico dominante
pca    = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(X_theta)

dom_topic     = theta.idxmax(axis=1).values
unique_topics = sorted(set(dom_topic), key=lambda t: int(t.split('_')[1]))

fig = go.Figure()
for t in unique_topics:
    mask   = dom_topic == t
    hover  = [f'{nombres.get(s, s)}<br>{s}' for s, m in zip(theta.index, mask) if m]
    fig.add_trace(go.Scatter(
        x=coords[mask, 0], y=coords[mask, 1],
        mode='markers',
        name=t,
        text=hover,
        hovertemplate='<b>%{text}</b><br>PC1: %{x:.3f}<br>PC2: %{y:.3f}<extra></extra>',
        marker=dict(size=9, opacity=0.85),
    ))

fig.update_layout(
    title=(
        f'PCA sobre theta — Cercanías (config óptima)<br>'
        f'<sup>PC1={pca.explained_variance_ratio_[0]*100:.1f}%  '
        f'PC2={pca.explained_variance_ratio_[1]*100:.1f}%  |  '
        f'n_topics={model.n_topics}  granularidad={best["token_granularity"]}  '
        f'use_periods={best["use_periods"]}</sup>'
    ),
    xaxis_title='PC1', yaxis_title='PC2',
    legend_title='Tópico dominante',
    height=620, hovermode='closest',
)
HTML(fig.to_html(include_plotlyjs='cdn', full_html=False))

In [ ]:
out_dir = Path('output')
out_dir.mkdir(exist_ok=True)

theta.to_parquet(out_dir / 'theta_cercanias_optuna.parquet')
theta.round(6).to_csv(out_dir / 'theta_cercanias_optuna.csv')
model.inspect_topics(15).to_csv(out_dir / 'topics_cercanias_optuna.csv')

df_trials = study.trials_dataframe()
df_trials.to_csv(out_dir / 'optuna_trials_lda_cercanias.csv', index=False)

print(f'Guardado en {out_dir.resolve()}')
print(f'  theta_cercanias_optuna.parquet      {theta.shape}')
print(f'  topics_cercanias_optuna.csv')
print(f'  optuna_trials_lda_cercanias.csv     {len(df_trials)} trials')
print(f'\n=== RESUMEN FINAL ===')
print(f'Score Optuna best (1 init):  {study.best_value:.4f}')
print(f'Score modelo final (3 init): {score_final:.4f}')
print(f'Tópicos activos:             {int(active_final * best["n_topics"])} / {best["n_topics"]}')
print(f'Vocab size:                  {model.X_.shape[1]} tokens')
print(f'Mejores hiperparámetros:')
for k, v in best.items():
    print(f'  {k:<22} = {v}')